# **Preprocessing the CERT Dataset**:

### Imports:

In [1]:
from scripts.Preprocessing import *

### Defining Constants and Defining CERT Path:

In [2]:
DEFAULT_OUTPUT_DIR = "processed_datasets"
WORK_HOURS = (9, 17)
LONG_URL_THRESHOLD = 90

In [3]:
cert_path = r"C:\Users\Hugo Marques\Documents\Insider-Threats\data\r6.2 (1)\r6.2"

### Building Layer A: (User, PC, Day) Level Dataset

In [ ]:
layer_a_dataset = build_layer_a(cert_path, WORK_HOURS)

,timestamp,user,pc,activity,day
2686,2010-01-04 07:41:00,aab0162,pc-6599,Logon,2010-01-04
10398,2010-01-04 18:46:00,aab0162,pc-6599,Logoff,2010-01-04
12559,2010-01-05 07:46:00,aab0162,pc-6599,Logon,2010-01-05
20419,2010-01-05 18:40:00,aab0162,pc-6599,Logoff,2010-01-05
22815,2010-01-06 07:45:00,aab0162,pc-6599,Logon,2010-01-06


In [ ]:
norm_files.head()

,timestamp,user,pc,filename,activity,to_removable_media,from_removable_media,content,day
44760,2010-01-13 10:34:04,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,False,False,25-50-44-46-2D If you look at the assassinatio...,2010-01-13
45574,2010-01-13 12:22:42,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,False,False,25-50-44-46-2D If you look at the assassinatio...,2010-01-13
65488,2010-01-18 15:40:26,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,False,False,25-50-44-46-2D If you look at the assassinatio...,2010-01-18
271634,2010-03-08 14:49:23,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,False,False,25-50-44-46-2D If you look at the assassinatio...,2010-03-08
571486,2010-05-19 13:10:35,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,False,False,25-50-44-46-2D If you look at the assassinatio...,2010-05-19


In [23]:
norm_devices = normalize_shared_columns(devices)

In [24]:
norm_devices.head()

,timestamp,user,pc,file_tree,activity,day
1404,2010-01-04 08:26:37,aac0610,pc-1834,R:\;R:\31j52f6;R:\44VQPl0;R:\52GqG81;R:\67Ppwr...,Connect,2010-01-04
2273,2010-01-04 10:23:08,aac0610,pc-1834,NaN,Disconnect,2010-01-04
8162,2010-01-05 14:10:01,aac0610,pc-1834,R:\;R:\31j52f6;R:\44VQPl0;R:\52GqG81;R:\67Ppwr...,Connect,2010-01-05
8241,2010-01-05 14:20:11,aac0610,pc-1834,NaN,Disconnect,2010-01-05
8619,2010-01-05 15:04:58,aac0610,pc-1834,R:\;R:\31j52f6;R:\44VQPl0;R:\52GqG81;R:\67Ppwr...,Connect,2010-01-05


In [25]:
norm_emails = normalize_shared_columns(emails)

In [26]:
norm_emails.head()

,timestamp,user,pc,to,cc,bcc,from,activity,size,attachments,content,day
6198,2010-01-04 08:34:26,aab0162,pc-6599,Tyrone.Axel.Prince@dtaa.com,Jeanette.Macey.Simpson@dtaa.com,NaN,Amos.Ahmed.Burch@dtaa.com,Send,27156,NaN,The forests and meadows of Bryce Canyon provid...,2010-01-04
6518,2010-01-04 08:40:32,aab0162,pc-6599,Amos.Ahmed.Burch@dtaa.com,NaN,NaN,Nicole_Moody@hp.com,View,27782,NaN,About 35% of Ember Ridge andesite contains phe...,2010-01-04
6541,2010-01-04 08:41:06,aab0162,pc-6599,Adria.Sylvia.Flowers@dtaa.com;Xenos.Akeem.Barr...,NaN,NaN,Amos.Ahmed.Burch@dtaa.com,Send,2351690,C:\AAB0162\8UXAP401.doc(2129531);C:\AAB0162\GL...,"So if you killed right wing figures, you'd als...",2010-01-04
6606,2010-01-04 08:42:09,aab0162,pc-6599,Amos.Ahmed.Burch@dtaa.com,NaN,NaN,Raphael.Adrian.Peck@dtaa.com,View,30821,NaN,"At that time, Harrison had never worked in Bri...",2010-01-04
6640,2010-01-04 08:42:48,aab0162,pc-6599,Amos.Ahmed.Burch@dtaa.com,NaN,NaN,WOB17@boeing.com,View,29378,NaN,Sioeng also joined Fong at a meeting with then...,2010-01-04


In [27]:
# Sanity checks
files = [norm_logons, norm_files, norm_devices, norm_emails]

for file in files:
    assert file["day"].dtype == "datetime64[ns]"
    assert set(["user", "pc", "timestamp", "day"]).issubset(norm_logons.columns)

### Feature Engineering Normalized Data:

In [28]:
def extract_logon_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily authentication behavior features from logon events to create an aggregated feature table.
    
    Args:
        norm_df: The normalized logon dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated logon behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)    
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving logon-related features
    features = grouped.agg(
        logon_count = ("activity", lambda x: (x=="Logon").sum()),
        logoff_count = ("activity", lambda x: (x=="Logoff").sum()),
        off_hours_logon = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [29]:
agg_logons = extract_logon_features(norm_logons)

In [30]:
agg_logons.head()

,user,pc,day,logon_count,logoff_count,off_hours_logon
0,aab0162,pc-6599,2010-01-04,1,1,2
1,aab0162,pc-6599,2010-01-05,1,1,2
2,aab0162,pc-6599,2010-01-06,1,1,2
3,aab0162,pc-6599,2010-01-07,1,1,2
4,aab0162,pc-6599,2010-01-08,1,1,2


In [31]:
def extract_file_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily file access behavior features to create an aggregated feature table.
    
    Args:
        norm_df: Normalized file activity dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated file behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving file-related features
    features = grouped.agg(
        file_open_count = ("activity", lambda x: (x=="File Open").sum()),
        file_write_count = ("activity", lambda x: (x=="File Write").sum()),
        file_copy_count = ("activity", lambda x: (x=="File Copy").sum()),
        file_delete_count = ("activity", lambda x: (x=="File Delete").sum()),
        unique_files_accessed = ("filename", "nunique"),
        off_hours_files_accessed = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [32]:
agg_files = extract_file_features(norm_files)

In [33]:
agg_files.head()

,user,pc,day,file_open_count,file_write_count,file_copy_count,file_delete_count,unique_files_accessed,off_hours_files_accessed
0,aab0162,pc-6599,2010-01-13,2,0,0,0,1,0
1,aab0162,pc-6599,2010-01-18,1,0,0,0,1,0
2,aab0162,pc-6599,2010-03-08,1,0,0,0,1,0
3,aab0162,pc-6599,2010-05-19,1,0,0,0,1,0
4,aab0162,pc-6599,2010-05-24,1,0,0,0,1,0


In [34]:
def extract_device_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily removable media (USB) behavior features to create an aggregated feature table.
    
    Args:
        norm_df: Normalized file activity dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated removable media behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving media-related features
    features = grouped.agg(
        usb_insert_count = ("activity", lambda x: (x=="Connect").sum()),
        usb_remove_count = ("activity", lambda x: (x=="Disconnect").sum()),
        off_hours_usb_usage = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [35]:
agg_devices = extract_device_features(norm_devices)

In [36]:
agg_devices.head()

,user,pc,day,usb_insert_count,usb_remove_count,off_hours_usb_usage
0,aac0610,pc-1834,2010-01-04,1,1,1
1,aac0610,pc-1834,2010-01-05,2,2,0
2,aac0610,pc-1834,2010-01-06,1,1,1
3,aac0610,pc-1834,2010-01-07,2,2,0
4,aac0610,pc-1834,2010-01-08,1,1,0


In [37]:
def extract_email_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily email communication behavior features to create an aggregated feature table.
    
    Args:
        norm_df: Normalized file activity dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated email behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # External email heuristic
    df["external_email"] = ~df["to"].str.contains("dtaa.com", na=False)
    
    # Creating groups based on (user, pc, day)
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving email-related features
    features = grouped.agg(
        emails_sent = ("to", "count"),
        unique_recipients = ("to", "nunique"),
        external_emails = ("external_email", "sum"),
        attachments_sent = ("attachments", lambda x: (x.notnull()).sum()),
        off_hours_emails = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [38]:
agg_emails = extract_email_features(norm_emails)

In [39]:
agg_emails.head()

,user,pc,day,emails_sent,unique_recipients,external_emails,attachments_sent,off_hours_emails
0,aab0162,pc-6599,2010-01-04,9,5,1,1,5
1,aab0162,pc-6599,2010-01-05,9,8,1,2,0
2,aab0162,pc-6599,2010-01-06,9,8,0,1,8
3,aab0162,pc-6599,2010-01-07,9,7,2,1,5
4,aab0162,pc-6599,2010-01-08,9,7,2,0,0


### Merging Aggregated Feature Tables Into One Behavioral Matrix:

In [40]:
def merge_behavorial_features(feature_tables: list[pd.DataFrame], merge_cols: list=["user", "pc", "day"]) -> pd.DataFrame:
    """
    Merges multiple aggreagated behavorial feature tables into a single dataset.
    
    Args:
        feature_tables: A list of aggreagated features dataframes to merge, where each table must contain the merge columns
        merge_cols: columns to merge on
        
    Returns:
        pd.DataFrame: A unified table with missing activitiy filled with zeros
    """
    # Filtering our empty tables
    valid_tables = [df for df in feature_tables if ((df is not None) and (not df.empty))]
    
    if not valid_tables:
        raise ValueError("No valid feature tables provided for merging")

    merged_df = valid_tables[0].copy()
    
    # Iteratively merging tables
    for df in valid_tables[1:]:
        merged_df = merged_df.merge(df, on=merge_cols, how="outer")
        
    # Identifying feature columns, excluding identifiers
    feature_cols = [col for col in merged_df.columns if col not in merge_cols]
    
    # Filling missing feature values with zero
    merged_df[feature_cols] = merged_df[feature_cols].fillna(0)
    merged_df[feature_cols] = merged_df[feature_cols].astype("int32")
    
    # Sorting dataframe for consistency
    merged_df.sort_values(by=merge_cols, inplace=True)
    merged_df.reset_index(drop=True, inplace=True)
    
    return merged_df

In [41]:
feature_tables = [agg_logons, agg_files, agg_devices, agg_emails]

In [42]:
behavioral_matrix = merge_behavorial_features(feature_tables)

In [43]:
behavioral_matrix

,user,pc,day,logon_count,logoff_count,off_hours_logon,file_open_count,file_write_count,file_copy_count,file_delete_count,unique_files_accessed,off_hours_files_accessed,usb_insert_count,usb_remove_count,off_hours_usb_usage,emails_sent,unique_recipients,external_emails,attachments_sent,off_hours_emails
0,aab0162,pc-6599,2010-01-04,1,1,2,0,0,0,0,0,0,0,0,0,9,5,1,1,5
1,aab0162,pc-6599,2010-01-05,1,1,2,0,0,0,0,0,0,0,0,0,9,8,1,2,0
2,aab0162,pc-6599,2010-01-06,1,1,2,0,0,0,0,0,0,0,0,0,9,8,0,1,8
3,aab0162,pc-6599,2010-01-07,1,1,2,0,0,0,0,0,0,0,0,0,9,7,2,1,5
4,aab0162,pc-6599,2010-01-08,1,1,2,0,0,0,0,0,0,0,0,0,9,7,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1564764,zzo2997,pc-3120,2011-05-24,1,1,1,0,0,0,0,0,0,0,0,0,14,11,7,1,0
1564765,zzo2997,pc-3120,2011-05-25,1,1,1,0,0,0,0,0,0,0,0,0,14,13,6,2,3
1564766,zzo2997,pc-3120,2011-05-26,1,1,1,0,0,0,0,0,0,0,0,0,14,13,6,2,0
1564767,zzo2997,pc-3120,2011-05-27,1,1,1,0,0,0,0,0,0,0,0,0,14,14,5,5,0


In [44]:
# Ensuring there are no duplicate rows
assert behavioral_matrix.duplicated(subset=["user", "pc", "day"]).sum() == 0

# Ensuring there are no empty cells in the feature space
assert behavioral_matrix.drop(columns=["user", "pc", "day"]).isna().sum().sum() == 0

### Feature Engineering PC-Related Data:

In [45]:
def extract_pc_features(df: pd.DataFrame, min_history: int=5) -> pd.DataFrame:
    """
    Extracts PC-derived contextual features to an aggregated behavioral matrix.
    
    Args:
        min_history: Minimum prior observations required before flagging new PC usage as abnormal activity
        
    Returns:
        pd.DataFrame: A behavioral matrix with added user-to-PC behavioral history
    """
    df = df.copy()
    df.sort_values(by=["user", "day", "pc"], inplace=True)
    
    # Tracking historical PC usage counts
    df["pc_prior_use_count"] = df.groupby(["user", "pc"]).cumcount()
    df["user_total_prior_days"] = df.groupby("user").cumcount()
    df["pc_seen_before"]  = (df["pc_prior_use_count"] > 0).astype(int)
    
    df["pc_prior_use_ratio"] = np.where(
        df["user_total_prior_days"] > 0,
        df["pc_prior_use_count"] / df["user_total_prior_days"],
        0.0
    )
    
    # Identifying a user's primary PC (i.e., most-used PC)
    primary_pc_map = (
        df.groupby(["user", "pc"])
        .size()
        .reset_index(name="count")
        .sort_values(["user", "count", "pc"], ascending=[True, False, True])
        .drop_duplicates(subset=["user"])
        .set_index("user")["pc"]
        .to_dict()
    )
    
    df["pc_is_primary"] = df.apply(
        lambda row: int(row["pc"] == primary_pc_map.get(row["user"])),
        axis=1
    )
    
    # Tracking the number of distinct PC's previously used
    df["distinct_pcs_prior"] = (
        df.groupby(["user"])["pc"]
        .transform(lambda s: [len(set(s.iloc[:i])) for i in range(len(s))])
    )
    
    # Tracking the number of unique PC's used on a given day
    same_day_counts = (
        df.groupby(["user", "day"])["pc"]
        .transform("nunique")
    )
    
    df["multi_pc_day"] = same_day_counts
    
    # Identifies new PC usage after an established history
    df["new_pc_after_history"] = (
        (df["pc_prior_use_count"] == 0) &
        (df["user_total_prior_days"] >= min_history)
    ).astype(int)
    
    df.drop(columns=["user_total_prior_days"], inplace=True)
    
    return df

In [46]:
pc_behavioral_matrix = extract_pc_features(behavioral_matrix)

In [47]:
layer_a_dataset.head()

,user,pc,day,logon_count,logoff_count,off_hours_logon,file_open_count,file_write_count,file_copy_count,file_delete_count,...,off_hours_http_requests,http_long_url_count,unique_domains_visited,pc_prior_use_count,pc_seen_before,pc_prior_use_ratio,pc_is_primary,distinct_pcs_used_prior,n_pcs_used_today,new_pc_after_stable_history
0,aab0162,pc-6599,2010-01-04,1,1,2,0,0,0,0,...,32,34,16,0,0,0.0,1,0,1,0
1,aab0162,pc-6599,2010-01-05,1,1,2,0,0,0,0,...,22,27,14,1,1,1.0,1,1,1,0
2,aab0162,pc-6599,2010-01-06,1,1,2,0,0,0,0,...,15,54,19,2,1,1.0,1,1,1,0
3,aab0162,pc-6599,2010-01-07,1,1,2,0,0,0,0,...,25,44,26,3,1,1.0,1,1,1,0
4,aab0162,pc-6599,2010-01-08,1,1,2,0,0,0,0,...,30,42,10,4,1,1.0,1,1,1,0


In [ ]:
layer_a_path = save_dataset(layer_a_dataset, "ueba_dataset_3a.csv", DEFAULT_OUTPUT_DIR)

Dataset successfully saved to: c:\Users\Hugo Marques\Documents\Insider-Threats\Insider-Threat-Detection\processed_datasets\ueba_dataset_3a_test.csv


### Building Layer B: (User, Day) Model Training Dataset

In [ ]:
layer_b_dataset = build_layer_b(layer_a_dataset, rolling_window=5)

In [ ]:
layer_b_dataset.head()

,user,day,logon_count,logoff_count,off_hours_logon,file_open_count,file_write_count,file_copy_count,file_delete_count,unique_files_accessed,...,non_primary_pc_http_requests_flag_rolling_delta,non_primary_pc_usb_flag_rolling_delta,non_primary_pc_file_copy_flag_rolling_delta,off_hours_activity_flag,usb_file_activity_flag,external_comm_activity_flag,jobsite_usb_activity_flag,suspicious_upload_flag,cloud_upload_flag,non_primary_pc_risk_flag
0,aab0162,2010-01-04,1,1,2,0,0,0,0,0,...,0.0,0.0,0.0,1,0,1,0,0,0,0
1,aab0162,2010-01-05,1,1,2,0,0,0,0,0,...,0.0,0.0,0.0,1,0,1,0,0,0,0
2,aab0162,2010-01-06,1,1,2,0,0,0,0,0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
3,aab0162,2010-01-07,1,1,2,0,0,0,0,0,...,0.0,0.0,0.0,1,0,1,0,0,0,0
4,aab0162,2010-01-08,1,1,2,0,0,0,0,0,...,0.0,0.0,0.0,1,0,1,0,0,0,0


In [ ]:
layer_b_path = save_dataset(layer_b_dataset, "ueba_dataset_3b.csv", DEFAULT_OUTPUT_DIR)

Dataset successfully saved to: c:\Users\Hugo Marques\Documents\Insider-Threats\Insider-Threat-Detection\processed_datasets\ueba_dataset_3b_test.csv


###  Utilizing Drill-Down Dataset Lookup

In [ ]:
# Example usage
user = "abh0349"
day = "2010-01-27"

In [ ]:
drill_down_table = get_drilldown(layer_a_dataset, user, day)

In [ ]:
ueba_matrix.head()

,user,pc,day,logon_count,logoff_count,off_hours_logon,file_open_count,file_write_count,file_copy_count,file_delete_count,...,off_hours_http_requests,http_long_url_count,unique_domains_visited,pc_prior_use_count,pc_seen_before,pc_prior_use_ratio,pc_is_primary,distinct_pcs_used_prior,n_pcs_used_today,new_pc_after_stable_history
11356,abh0349,pc-1019,2010-01-27,2,1,1,0,0,0,0,...,2,7,7,17,1,0.809524,1,5,2,0
11357,abh0349,pc-1534,2010-01-27,1,1,0,0,0,0,0,...,0,0,0,0,0,0.000000,0,5,2,1


### Saving the Final UEBA-Enhanced Dataset:

In [51]:
def save_dataset(dataset: pd.DataFrame, filename: str="ueba_dataset.csv") -> None:
    """
    Saves the final UEBA-enhanced dataset to the specified path as a CSV file.
    
    Args:
        dataset: The UEBA-enhanced dataset
        file_name: The desired name of the CSV dataset
        
    Returns:
        None:
    """
    # Ensures directory exists
    save_path = os.path.join(os.getcwd(), "processed_datasets")
    os.makedirs(save_path, exist_ok=True)
    
    # Creates full file path
    file_path = os.path.join(save_path, filename)
    
    # Saving the dataset
    dataset.to_csv(file_path)
    print(f"Dataset successfully saved to: {file_path}")

In [52]:
save_dataset(ueba_matrix, "ueba_dataset_2.csv")

Dataset successfully saved to: c:\Users\Matt\Documents\Repos\Insider-Threat-Detection\processed_datasets\ueba_dataset_2.csv
